# Multi-Agent Task Decomposition with `asyncio.gather`

This notebook demonstrates how to break a complex task into parallel subtasks using multiple OpenAI API calls with `asyncio.gather`, then synthesize the results — all using the standard `openai` SDK (no Agents SDK required).

**What you will learn:**
- The decomposition pattern: one complex task → N independent specialist calls → synthesis
- How `asyncio.gather` runs API calls in parallel, cutting wall-clock time dramatically
- Handling partial failures gracefully with `return_exceptions=True`
- How to scale to more specialists without changing the gather logic

**Models used:**
- `gpt-4o-mini` — fast, cheap specialist agents
- `gpt-4o` — higher-quality synthesis

> **No hardcoded keys.** `AsyncOpenAI()` reads `OPENAI_API_KEY` automatically from your environment.

## 1. Setup

Install the `openai` package if it is not already present, then import everything we need.

In [ ]:
%pip install openai --quiet

In [ ]:
import asyncio
import json
import time
from openai import AsyncOpenAI

# Defer AsyncOpenAI() to run_pipeline() so mock-only runs need no API key.
# Create client only when needed (set to None for mock-only runs)
client: AsyncOpenAI | None = None

SPECIALIST_MODEL = "gpt-4o-mini"
SYNTHESIS_MODEL  = "gpt-4o"

print("Imports OK — client will be created on first live call.")

## 2. The Decomposition Pattern

```
Complex Task
     │
     ▼
┌─────────────────────────────────────────────┐
│           Decompose into N subtasks         │
└─────────────────────────────────────────────┘
     │           │           │           │
     ▼           ▼           ▼           ▼
 Specialist A  Specialist B  Specialist C  Specialist D
 (async call)  (async call)  (async call)  (async call)
     │           │           │           │
     └───────────┴───────────┴───────────┘
                       │
               asyncio.gather()
                       │
                       ▼
              ┌─────────────────┐
              │   Synthesizer   │
              │  (gpt-4o call)  │
              └─────────────────┘
                       │
                       ▼
              Final coherent report
```

**Key insight:** the N specialist calls are **independent** — none needs the output of another. That makes them safe to run in parallel with `asyncio.gather`, which fires all coroutines concurrently and waits for all to finish. Total wall-clock time ≈ the slowest single call, not the sum of all calls.

**When this pattern fits:**
- Each subtask can be answered without knowing what the others return
- Subtask outputs are *additive* (they get stitched together, not chained)
- You want specialised system prompts per role (historian, critic, forecaster, …)

**When it does NOT fit:**
- Subtask B needs subtask A's answer as input (use a sequential chain instead)
- The full answer requires one coherent reasoning chain across all parts

## 3. Concrete Use-Case — Research Report

Given a **topic**, we decompose the research task into four independent angles:

| Specialist | What it covers |
|---|---|
| Historian | Historical background and timeline |
| SOTA Analyst | Current state of the art |
| Challenges Analyst | Key open challenges |
| Futurist | Future directions and predictions |

These four angles are independent — none needs the other's answer — so we can gather them all at once.

## 4. Specialist System Prompts (module-level constants)

In [ ]:
HISTORIAN_PROMPT = """
You are a historian specialising in science and technology.
Given a topic, write a concise historical background section (150-200 words).
Cover the origins, key milestones, and pivotal moments that shaped where the field is today.
Use clear, accessible language suitable for a general technical audience.
""".strip()

SOTA_PROMPT = """
You are a research analyst tracking the state of the art in science and technology.
Given a topic, write a concise current-state-of-the-art section (150-200 words).
Highlight leading approaches, recent breakthroughs, and the best-performing methods or tools today.
Be specific: name techniques, benchmarks, or landmark papers where relevant.
""".strip()

CHALLENGES_PROMPT = """
You are a critical analyst who identifies open problems in science and technology.
Given a topic, write a concise key-challenges section (150-200 words).
Cover technical, ethical, societal, and resource barriers that remain unsolved.
Frame challenges as concrete blockers, not vague concerns.
""".strip()

FUTURE_PROMPT = """
You are a technology forecaster with expertise in emerging science and engineering.
Given a topic, write a concise future-directions section (150-200 words).
Identify promising research directions, likely paradigm shifts, and near-term vs long-term horizons.
Ground predictions in current trends rather than pure speculation.
""".strip()

SPECIALIST_REGISTRY = {
    "historical_background": HISTORIAN_PROMPT,
    "state_of_the_art":      SOTA_PROMPT,
    "key_challenges":        CHALLENGES_PROMPT,
    "future_directions":     FUTURE_PROMPT,
}

print(f"Registered {len(SPECIALIST_REGISTRY)} specialists: {list(SPECIALIST_REGISTRY)}")

## 5. Core Async Wrapper — `call_specialist`

A single async function that makes one API call. We give it a `timeout` parameter so a slow call does not block the gather indefinitely.

In [ ]:
async def call_specialist(
    client: AsyncOpenAI,
    system_prompt: str,
    topic: str,
    *,
    model: str = SPECIALIST_MODEL,
    timeout: float = 60.0,
) -> str:
    """
    Make a single async chat-completion call for one specialist role.

    Parameters
    ----------
    client        : AsyncOpenAI instance (reads OPENAI_API_KEY from env)
    system_prompt : Role-defining system message for this specialist
    topic         : The research topic passed as the user message
    model         : Model to use (default gpt-4o-mini)
    timeout       : Max seconds to wait before raising asyncio.TimeoutError

    Returns
    -------
    str : The specialist's response text
    """
    response = await asyncio.wait_for(
        client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": f"Topic: {topic}"},
            ],
            temperature=0.4,
            max_tokens=400,
        ),
        timeout=timeout,
    )
    return response.choices[0].message.content.strip()


print("call_specialist defined.")

## 6. Parallel Gather — `decompose_and_gather`

`asyncio.gather` fires all four coroutines **simultaneously** and collects results in the same order they were submitted. Total latency ≈ slowest single call, not 4× a single call.

In [ ]:
async def decompose_and_gather(
    topic: str,
    specialist_registry: dict[str, str] | None = None,
    client: AsyncOpenAI | None = None,
) -> dict[str, str]:
    """
    Decompose `topic` into parallel specialist calls and gather results.

    Parameters
    ----------
    topic               : Research topic string
    specialist_registry : Mapping of role_name -> system_prompt.
                          Defaults to SPECIALIST_REGISTRY (4 roles).
    client              : AsyncOpenAI client; creates one if not provided.

    Returns
    -------
    dict[str, str] : Mapping of role_name -> specialist output text
    """
    if client is None:
        client = AsyncOpenAI()
    if specialist_registry is None:
        specialist_registry = SPECIALIST_REGISTRY

    role_names = list(specialist_registry.keys())
    coroutines = [
        call_specialist(client, system_prompt, topic)
        for system_prompt in specialist_registry.values()
    ]

    # All coroutines fire in parallel here.
    results = await asyncio.gather(*coroutines)

    return dict(zip(role_names, results))


print("decompose_and_gather defined.")

## 7. Synthesis — `synthesize_report`

After gathering all specialist outputs, we pass them to a synthesizer that weaves them into one coherent report. We use `gpt-4o` here for higher-quality output.

In [ ]:
def build_synthesis_prompt(section_names: list[str]) -> str:
    """Build a synthesis system prompt from the actual section list."""
    numbered = "\n".join(
        f"  {i}. {name.replace('_', ' ').title()}"
        for i, name in enumerate(section_names, 1)
    )
    return f"""You are a senior research editor.
You receive {len(section_names)} independently written sections of a research brief:
{numbered}

Your job: weave these sections into ONE coherent, well-structured research report.
- Preserve the key insights from every section.
- Add smooth transitions between sections.
- Write an executive summary paragraph at the top (2-3 sentences).
- Use clear Markdown headers.
- Total length: 600-800 words.
- If a section is marked UNAVAILABLE, note the gap without fabricating content.""".strip()


async def synthesize_report(
    client: AsyncOpenAI,
    topic: str,
    results: dict[str, str],
) -> str:
    """
    Synthesize four specialist outputs into a single coherent research report.

    Parameters
    ----------
    client  : AsyncOpenAI instance
    topic   : Original research topic
    results : dict mapping role_name -> specialist text (use 'UNAVAILABLE' for failed sections)

    Returns
    -------
    str : Final synthesized report in Markdown
    """
    # Build a structured user message that presents all specialist outputs.
    sections_text = "\n\n".join(
        f"### {role.replace('_', ' ').title()}\n{text}"
        for role, text in results.items()
    )
    user_message = f"Topic: {topic}\n\n{sections_text}"

    system_prompt = build_synthesis_prompt(list(results.keys()))
    response = await client.chat.completions.create(
        model=SYNTHESIS_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.3,
        max_tokens=1200,
    )
    return response.choices[0].message.content.strip()


print("synthesize_report defined.")

## 8. Mock Demo — Full Pipeline End-to-End

The cells below use **mock outputs** so you can run the notebook without an API key and see the full pipeline in action. Switch `USE_MOCK = False` to make live API calls.

In [ ]:
USE_MOCK = True   # Set to False to make real API calls (requires OPENAI_API_KEY)
TOPIC = "Large Language Models"

# ---------------------------------------------------------------------------
# Mock specialist outputs — realistic placeholders
# ---------------------------------------------------------------------------
MOCK_RESULTS = {
    "historical_background": (
        "Large Language Models trace their roots to early statistical language models of the 1990s. "
        "The introduction of neural language models (Bengio et al., 2003) marked a turning point, "
        "followed by recurrent networks (LSTMs) that dominated the 2010s. The watershed moment came "
        "with 'Attention Is All You Need' (Vaswani et al., 2017), which introduced the Transformer "
        "architecture. GPT-1 (2018) and BERT (2018) demonstrated that large-scale pre-training on "
        "raw text could transfer to many downstream tasks. GPT-3 (2020) showed emergent few-shot "
        "capabilities, and subsequent models — PaLM, LLaMA, Claude, Gemini — have pushed scale, "
        "efficiency, and alignment further."
    ),
    "state_of_the_art": (
        "As of 2025, frontier LLMs achieve near-human performance on many reasoning benchmarks. "
        "GPT-4o, Claude 3.5 Sonnet, and Gemini 1.5 Pro lead on multi-modal tasks. Open-weight "
        "models such as Llama 3 70B and Mistral 7B have closed much of the capability gap. "
        "RLHF and Constitutional AI are standard alignment techniques. Long-context windows (up to "
        "1M tokens) enable document-level reasoning. Mixture-of-Experts (MoE) architectures like "
        "Mixtral deliver frontier quality at reduced inference cost. Retrieval-Augmented Generation "
        "(RAG) is the dominant pattern for grounding models in up-to-date external knowledge."
    ),
    "key_challenges": (
        "Key open challenges include: (1) Hallucination — models confidently generate factually "
        "incorrect text with no reliable self-correction mechanism. (2) Reasoning reliability — "
        "performance on novel multi-step math and logic tasks remains brittle. (3) Alignment and "
        "safety — preventing misuse, jailbreaks, and deceptive behaviour at scale is unsolved. "
        "(4) Compute cost — training and serving frontier models requires enormous energy and "
        "capital, concentrating power among a few organisations. (5) Data quality and bias — "
        "pre-training corpora encode societal biases that propagate into model outputs. "
        "(6) Evaluation — existing benchmarks saturate quickly, making it hard to measure true "
        "progress."
    ),
    "future_directions": (
        "Near-term (1-3 years): wider deployment of multi-modal models handling text, image, audio, "
        "and video; tighter integration with external tools and agentic workflows; more efficient "
        "inference via quantisation and speculative decoding. Medium-term (3-7 years): models that "
        "learn continuously from user interactions without catastrophic forgetting; formal "
        "verification of model outputs for safety-critical domains; smaller, specialised models "
        "rivalling large general models on domain benchmarks. Long-term: potential paradigm shifts "
        "toward neurosymbolic hybrids, models with persistent memory, and fundamentally new "
        "training objectives beyond next-token prediction."
    ),
}

MOCK_SYNTHESIS = """
## Research Report: Large Language Models

**Executive Summary.** Large Language Models have evolved from narrow statistical tools into 
general-purpose reasoning systems that approach human-level performance on a broad range of tasks. 
While the field has made remarkable progress since the Transformer's introduction in 2017, 
fundamental challenges around reliability, alignment, and compute cost remain open research problems.

### Historical Background
Large Language Models trace their roots to early statistical language models of the 1990s. The 
watershed moment came with the Transformer architecture (2017), followed by GPT-3's demonstration 
of emergent few-shot capabilities in 2020.

### State of the Art
Frontier models such as GPT-4o, Claude 3.5 Sonnet, and open-weight Llama 3 70B now support 
million-token context windows, multimodal inputs, and sophisticated tool use. 
Mixture-of-Experts architectures have made frontier-quality inference economically viable at scale.

### Key Challenges
Hallucination, brittle multi-step reasoning, alignment at scale, and concentration of compute 
resources are the four most pressing blockers to safe, equitable deployment.

### Future Directions
The next decade is likely to bring continuous-learning models, neurosymbolic hybrids, 
and a shift away from pure next-token prediction toward objectives that better capture 
world knowledge and causal reasoning.
""".strip()


async def run_pipeline(topic: str, use_mock: bool = True):
    """Full pipeline: decompose → gather → synthesize."""
    if use_mock:
        print("[MOCK MODE] Using pre-written specialist outputs.\n")
        gathered = MOCK_RESULTS
        report   = MOCK_SYNTHESIS
    else:
        c = AsyncOpenAI()
        print(f"Gathering specialist outputs for topic: '{topic}' ...")
        gathered = await decompose_and_gather_safe(topic, client=c)
        print("Synthesizing report ...")
        report   = await synthesize_report(c, topic, gathered)

    print("=" * 60)
    print("SPECIALIST OUTPUTS")
    print("=" * 60)
    for role, text in gathered.items():
        print(f"\n--- {role.replace('_', ' ').upper()} ---")
        print(text[:300], "..." if len(text) > 300 else "")

    print("\n" + "=" * 60)
    print("SYNTHESIZED REPORT")
    print("=" * 60)
    print(report)
    return gathered, report


gathered, report = await run_pipeline(TOPIC, use_mock=USE_MOCK)

## 9. Timing Comparison — Sequential vs Parallel

This cell **simulates** the timing difference using `asyncio.sleep` instead of real API calls, so you can observe the speedup without a key.

In [ ]:
import time

SIMULATED_CALL_SECONDS = 2.0   # pretend each specialist call takes 2 s
N_SPECIALISTS = 4


async def fake_specialist_call(role: str, delay: float) -> str:
    """Simulate a specialist API call with a fixed delay."""
    await asyncio.sleep(delay)
    return f"[{role}] mock output after {delay:.1f}s"


# --- Sequential (one call at a time) ---
t0 = time.perf_counter()
sequential_results = []
for role in SPECIALIST_REGISTRY:
    result = await fake_specialist_call(role, SIMULATED_CALL_SECONDS)
    sequential_results.append(result)
sequential_time = time.perf_counter() - t0

# --- Parallel (asyncio.gather) ---
t0 = time.perf_counter()
parallel_results = await asyncio.gather(
    *[fake_specialist_call(role, SIMULATED_CALL_SECONDS) for role in SPECIALIST_REGISTRY]
)
parallel_time = time.perf_counter() - t0

print(f"Simulated single-call latency : {SIMULATED_CALL_SECONDS:.1f}s")
print(f"Sequential time ({N_SPECIALISTS} calls in series) : {sequential_time:.2f}s")
print(f"Parallel time   ({N_SPECIALISTS} calls via gather) : {parallel_time:.2f}s")
print(f"Speedup factor  : {sequential_time / parallel_time:.1f}x")
print()
print("Note: with real API calls the speedup is typically 3-4x for 4 specialists,")
print("because network latency and server processing overlap.")

## 10. Error Handling — Partial Failures with `return_exceptions=True`

In production, one specialist call may time out or hit a rate limit while the others succeed. Using `return_exceptions=True` prevents a single failure from cancelling all other tasks. We inspect the results and substitute a placeholder for any failed section.

In [ ]:
async def decompose_and_gather_safe(
    topic: str,
    specialist_registry: dict[str, str] | None = None,
    client: AsyncOpenAI | None = None,
) -> dict[str, str]:
    """
    Fault-tolerant version of decompose_and_gather.

    Uses return_exceptions=True so that a single failed specialist call
    does not cancel the others. Failed sections are replaced with
    'UNAVAILABLE (<error type>)' so the synthesizer can handle them gracefully.
    """
    if client is None:
        client = AsyncOpenAI()
    if specialist_registry is None:
        specialist_registry = SPECIALIST_REGISTRY

    role_names = list(specialist_registry.keys())
    coroutines = [
        call_specialist(client, system_prompt, topic)
        for system_prompt in specialist_registry.values()
    ]

    # return_exceptions=True: exceptions are returned as values, not raised.
    raw = await asyncio.gather(*coroutines, return_exceptions=True)

    results: dict[str, str] = {}
    for role, value in zip(role_names, raw):
        if isinstance(value, Exception):
            error_label = type(value).__name__
            print(f"WARNING: specialist '{role}' failed with {error_label}: {value}")
            results[role] = f"UNAVAILABLE ({error_label})"
        else:
            results[role] = value

    failed = [r for r, v in results.items() if v.startswith("UNAVAILABLE")]
    if failed:
        print(f"\n{len(failed)} section(s) unavailable: {failed}")
        print("The synthesizer will note these gaps in the final report.")
    else:
        print("All specialists returned successfully.")

    return results


# --- Demo: simulate one specialist failing ---
async def flaky_specialist(client, system_prompt, topic):
    """Intentionally fails to simulate a timeout."""
    raise asyncio.TimeoutError("simulated timeout after 60s")



async def _mock_coro(value):
    """Return a mock value as a coroutine (asyncio.coroutine was removed in 3.11)."""
    return value

# Inject the flaky coroutine into a test gather
role_names_demo = list(SPECIALIST_REGISTRY.keys())
demo_coroutines = [
    flaky_specialist(None, None, TOPIC),   # fails
    _mock_coro(MOCK_RESULTS["state_of_the_art"]),
    _mock_coro(MOCK_RESULTS["key_challenges"]),
    _mock_coro(MOCK_RESULTS["future_directions"]),
]
raw_demo = await asyncio.gather(*demo_coroutines, return_exceptions=True)

safe_results: dict[str, str] = {}
for role, value in zip(role_names_demo, raw_demo):
    if isinstance(value, Exception):
        print(f"WARNING: '{role}' failed — {type(value).__name__}")
        safe_results[role] = f"UNAVAILABLE ({type(value).__name__})"
    else:
        safe_results[role] = value

print("\nFault-tolerant results:")
for role, val in safe_results.items():
    status = "FAILED" if val.startswith("UNAVAILABLE") else "OK"
    print(f"  {role:<25} -> {status}")

## 11. Scaling Up — Adding More Specialists

The gather logic does not need to change when you add new specialists. Just extend `SPECIALIST_REGISTRY` with a new role name and system prompt, and `decompose_and_gather` will pick it up automatically.

In [ ]:
# --- Example: add two new specialist roles ---

ETHICS_PROMPT = """
You are an AI ethics researcher.
Given a topic, write a concise ethical and societal implications section (150-200 words).
Cover fairness, transparency, accountability, privacy, and risk of misuse.
""".strip()

INDUSTRY_PROMPT = """
You are an industry analyst covering commercial applications of technology.
Given a topic, write a concise industry adoption section (150-200 words).
Identify leading companies, major products, market size estimates, and adoption barriers.
""".strip()

# Extend the registry — no other code changes needed.
EXTENDED_REGISTRY = {
    **SPECIALIST_REGISTRY,             # original 4
    "ethics_and_society": ETHICS_PROMPT,
    "industry_adoption":  INDUSTRY_PROMPT,
}

print(f"Extended registry has {len(EXTENDED_REGISTRY)} specialists:")
for i, role in enumerate(EXTENDED_REGISTRY, 1):
    print(f"  {i}. {role}")

print()
print("decompose_and_gather(topic, specialist_registry=EXTENDED_REGISTRY)")
print("  -> fires 6 parallel calls instead of 4, no other code changes required.")
print()
print("Estimated parallel wall-clock time: still ≈ slowest single call (~2-4s)")
print("Estimated sequential wall-clock time would be: 6 × ~3s = ~18s")

## 12. Summary — When to Use Decomposition vs Single Prompt

| Criterion | Use Decomposition + Gather | Use Single Prompt |
|---|---|---|
| Subtask independence | Subtasks are fully independent | Each step depends on the previous answer |
| Output structure | Additive sections that get stitched together | One continuous reasoning chain |
| Specialisation | Each role benefits from a targeted system prompt | A single expert persona is sufficient |
| Latency budget | Low — parallel calls cut wall time to ≈ slowest call | Irrelevant — only one call needed |
| Cost | Higher (N calls) — worth it for long outputs | Lower — one call |
| Error tolerance | Can degrade gracefully if one specialist fails | All-or-nothing |
| Context window | Subtasks fit in separate contexts | Full context fits in one prompt |

**Rule of thumb:** if you can draw a diagram where the subtasks sit in parallel (not in series), use decomposition. If the diagram is a chain, use a sequential pipeline or a single complex prompt.

---

### Key Takeaways

1. `asyncio.gather(*coroutines)` is the single most important line in this pattern — it is what makes the calls parallel.
2. Use `return_exceptions=True` in production so one failure does not cancel everything.
3. Specialist system prompts keep each call focused; the synthesizer sees all outputs and does the integration.
4. Adding more specialists costs **zero refactoring** — just extend the registry dict.
5. Use `gpt-4o-mini` for specialists (cheap, fast) and `gpt-4o` for synthesis (higher quality where it matters).

---

### Next Steps

- **Structured outputs:** have each specialist return JSON so the synthesizer can reference fields programmatically.
- **Streaming synthesis:** stream the `gpt-4o` synthesis call so the user sees partial output as it arrives.
- **Caching:** cache specialist results by `(topic, role)` key to avoid redundant API calls on repeat queries.
- **Dynamic decomposition:** let a planning call decide which specialists to invoke based on the topic, rather than hard-coding the registry.